# Adaptive Adversarial Threat Detection and Defense Framework
## Complete Experiment Notebook

Runs the full experiment end-to-end:
- **Part 1** — Load & Preprocess NSL-KDD data
- **Part 2** — Baseline Random Forest (Condition 1 / C1)
- **Part 3** — Adversarial Attack Simulation (Condition 2 / C2)
- **Part 4** — Adversarial Training Defense (Condition 3 / C3)
- **Part 5** — Three-Condition Comparison

**7 figures** are saved as PNG files in the Colab working directory (`/content/`).

---

### Before running: get the NSL-KDD dataset
1. Go to **https://www.unb.ca/cic/datasets/nsl.html**
2. Download `KDDTrain+.txt` and `KDDTest+.txt`
3. Upload both files using the cell below (or drag them into the Files panel on the left)


In [ ]:
# ── Upload NSL-KDD files ────────────────────────────────────────────────────
# Run this cell to open a file picker dialog in Google Colab.
# Select BOTH KDDTrain+.txt and KDDTest+.txt at once.
# Skip this cell if you already uploaded the files via the Files panel.
try:
    from google.colab import files
    print('Click "Choose Files" and select KDDTrain+.txt and KDDTest+.txt')
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))
except ImportError:
    print('Not running in Google Colab.')
    print('Ensure KDDTrain+.txt and KDDTest+.txt are in your working directory.')


---
## Part 1 — Imports, Data Loading & Preprocessing


In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('All imports successful.')


In [ ]:
# ── Column names (NSL-KDD has no header row) ────────────────────────────────
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty'
]

train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df  = pd.read_csv('KDDTest+.txt',  names=columns)
print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')
print(f'Null values — Train: {train_df.isnull().sum().sum()}  |  Test: {test_df.isnull().sum().sum()}')
train_df.head(3)


In [ ]:
# ── Preprocessing (all 5 steps) ─────────────────────────────────────────────

# Step 1: Drop difficulty — it is a meta-attribute, not a network traffic feature
train_df = train_df.drop('difficulty', axis=1)
test_df  = test_df.drop('difficulty', axis=1)

# Step 2: Binary labels — normal=0, all attack types=1
train_df['label'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['label']  = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
print('Label distribution after binarisation:')
print(f'  Train — normal: {(train_df["label"]==0).sum():>6}  attack: {(train_df["label"]==1).sum():>6}')
print(f'  Test  — normal: {(test_df["label"]==0).sum():>6}  attack: {(test_df["label"]==1).sum():>6}')

# Step 3: Encode categoricals
# Fit on combined train+test so test-set categories absent from training are handled
cat_cols = ['protocol_type', 'service', 'flag']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train_df[col], test_df[col]]))
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])
    encoders[col] = le
print('Categorical encoding complete.')

# Step 4: Separate features and labels (use the benchmark split — no train_test_split)
feature_cols = [c for c in train_df.columns if c != 'label']
X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test  = test_df[feature_cols].values
y_test  = test_df['label'].values

# Step 5: Normalize — ONLY fit on X_train to prevent data leakage
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'\nX_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'X_train mean (should be ~0): {X_train.mean():.6f}')
print(f'X_train std  (should be ~1): {X_train.std():.6f}')
print('Preprocessing complete.')


---
## Part 2 — Condition 1: Baseline Random Forest


In [ ]:
# ── Train baseline Random Forest ────────────────────────────────────────────
# n_jobs=-1 uses all CPU cores. This takes ~1-2 minutes on Colab free tier.
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)
print('Baseline Random Forest training complete.')


In [ ]:
# ── Evaluate on clean test set (Condition 1 — Baseline) ─────────────────────
y_pred_baseline = rf_baseline.predict(X_test)
y_prob_baseline = rf_baseline.predict_proba(X_test)[:, 1]

acc_c1  = accuracy_score(y_test, y_pred_baseline)
prec_c1 = precision_score(y_test, y_pred_baseline)
rec_c1  = recall_score(y_test, y_pred_baseline)
f1_c1   = f1_score(y_test, y_pred_baseline)
auc_c1  = roc_auc_score(y_test, y_prob_baseline)

print('=== CONDITION 1: BASELINE RESULTS ===')
print(f'Accuracy : {acc_c1:.4f}')
print(f'Precision: {prec_c1:.4f}')
print(f'Recall   : {rec_c1:.4f}')
print(f'F1-Score : {f1_c1:.4f}')
print(f'AUC-ROC  : {auc_c1:.4f}')


In [ ]:
# ── Feature importances — identify top-5 attack targets ─────────────────────
importances = rf_baseline.feature_importances_
top10_idx   = np.argsort(importances)[::-1][:10]
top5_idx    = top10_idx[:5]   # used in Part 3 and Part 4

print('Top 10 features by importance:')
for rank, idx in enumerate(top10_idx, 1):
    marker = ' <-- attack target' if idx in top5_idx else ''
    print(f'  {rank:2}. {feature_cols[idx]:38s} {importances[idx]:.4f}{marker}')

print(f'\ntop5_idx = {top5_idx.tolist()}')
print(f'Top-5 names: {[feature_cols[i] for i in top5_idx]}')


In [ ]:
# ── Figure 1: Confusion matrix — C1 Baseline ────────────────────────────────
cm_c1 = confusion_matrix(y_test, y_pred_baseline)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_c1, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title('Figure 1 — Confusion Matrix: C1 Baseline')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion-matrix-baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion-matrix-baseline.png')


In [ ]:
# ── Figure 2: ROC curve — C1 Baseline ───────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob_baseline)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'C1 Baseline (AUC = {auc_c1:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Figure 2 — ROC Curve: C1 Baseline')
plt.legend()
plt.tight_layout()
plt.savefig('roc-baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc-baseline.png')


In [ ]:
# ── Figure 3: Feature importance bar chart ───────────────────────────────────
top10_names  = [feature_cols[i] for i in top10_idx]
top10_values = importances[top10_idx]
colors = ['crimson' if i in top5_idx else 'steelblue' for i in top10_idx]

plt.figure(figsize=(9, 5))
bars = plt.barh(top10_names[::-1], top10_values[::-1],
                color=colors[::-1])
plt.xlabel('Importance Score')
plt.title('Figure 3 — Top 10 Feature Importances (red = adversarial attack targets)')
plt.tight_layout()
plt.savefig('feature-importance-baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature-importance-baseline.png')


In [ ]:
# ── C1 results summary — copy these values to results-baseline.md ────────────
print('=' * 55)
print('  CONDITION 1 RESULTS  —  copy to results-baseline.md')
print('=' * 55)
print(f'  Accuracy : {acc_c1:.4f}')
print(f'  Precision: {prec_c1:.4f}')
print(f'  Recall   : {rec_c1:.4f}')
print(f'  F1-Score : {f1_c1:.4f}')
print(f'  AUC-ROC  : {auc_c1:.4f}')
print('-' * 55)
print(f'  top5_idx : {top5_idx.tolist()}')
print(f'  Top-5    : {[feature_cols[i] for i in top5_idx]}')
print('=' * 55)


---
## Part 3 — Condition 2: Adversarial Attack Simulation

Gaussian noise (mean=0) is added to the top-5 features of **attack-class test samples only**.
Three noise levels are tested: std = 0.1, 0.3, 0.5.


In [ ]:
# ── Adversarial test-set generator ──────────────────────────────────────────
def create_adversarial_test(X_test, y_test, top5_idx, noise_std, random_state=42):
    """
    Add Gaussian noise to the top-5 features of attack-class test samples.
    Normal-class samples are not modified.
    """
    np.random.seed(random_state)
    X_adv       = X_test.copy()
    attack_rows = np.where(y_test == 1)[0]
    noise       = np.random.normal(0, noise_std,
                                   size=(len(attack_rows), len(top5_idx)))
    # np.ix_ indexes specific rows AND columns simultaneously
    X_adv[np.ix_(attack_rows, top5_idx)] += noise
    return X_adv

print('create_adversarial_test() defined.')


In [ ]:
# ── Run attack at 3 noise levels ─────────────────────────────────────────────
noise_levels   = [0.1, 0.3, 0.5]
attack_results = {}

for std in noise_levels:
    X_adv      = create_adversarial_test(X_test, y_test, top5_idx, noise_std=std)
    y_pred_adv = rf_baseline.predict(X_adv)
    y_prob_adv = rf_baseline.predict_proba(X_adv)[:, 1]
    attack_results[std] = {
        'accuracy': accuracy_score(y_test, y_pred_adv),
        'recall':   recall_score(y_test, y_pred_adv),
        'f1':       f1_score(y_test, y_pred_adv),
        'auc':      roc_auc_score(y_test, y_prob_adv)
    }

print('=== CONDITION 2: ADVERSARIAL ATTACK RESULTS ===')
print(f'{"Noise std":>10} | {"Accuracy":>10} | {"Recall":>8} | {"F1":>8} | {"AUC":>8} | {"Recall Drop":>11}')
print('-' * 70)
print(f'{"C1 (clean)":>10} | {acc_c1:>10.4f} | {rec_c1:>8.4f} | {f1_c1:>8.4f} | {auc_c1:>8.4f} | {"baseline":>11}')
for std, m in attack_results.items():
    drop = rec_c1 - m['recall']
    print(f'{std:>10.1f} | {m["accuracy"]:>10.4f} | {m["recall"]:>8.4f} | {m["f1"]:>8.4f} | {m["auc"]:>8.4f} | {drop:>+11.4f}')


In [ ]:
# ── Select defense noise level ────────────────────────────────────────────────
# Default: 0.3 (medium — meaningful but non-catastrophic recall drop).
# Change this only if your results above justify it:
#   std=0.3 drop < 0.05  →  try 0.5
#   std=0.3 drop > 0.50  →  try 0.1
DEFENSE_NOISE_STD = 0.3

X_adv_selected = create_adversarial_test(X_test, y_test, top5_idx,
                                          noise_std=DEFENSE_NOISE_STD)
y_pred_c2 = rf_baseline.predict(X_adv_selected)
y_prob_c2 = rf_baseline.predict_proba(X_adv_selected)[:, 1]

acc_c2  = accuracy_score(y_test, y_pred_c2)
rec_c2  = recall_score(y_test, y_pred_c2)
f1_c2   = f1_score(y_test, y_pred_c2)
auc_c2  = roc_auc_score(y_test, y_prob_c2)

print(f'Selected DEFENSE_NOISE_STD = {DEFENSE_NOISE_STD}')
print(f'C2 Accuracy : {acc_c2:.4f}')
print(f'C2 Recall   : {rec_c2:.4f}  (drop from C1: {rec_c1 - rec_c2:+.4f})')
print(f'C2 F1-Score : {f1_c2:.4f}')
print(f'C2 AUC-ROC  : {auc_c2:.4f}')


In [ ]:
# ── Figure 4: Attack curve — recall and F1 vs. noise level ───────────────────
noise_x   = [0.0, 0.1, 0.3, 0.5]
recall_y  = [rec_c1] + [attack_results[s]['recall'] for s in [0.1, 0.3, 0.5]]
f1_y      = [f1_c1]  + [attack_results[s]['f1']     for s in [0.1, 0.3, 0.5]]

plt.figure(figsize=(8, 5))
plt.plot(noise_x, recall_y, marker='o', color='crimson',    lw=2, label='Recall')
plt.plot(noise_x, f1_y,     marker='s', color='steelblue',  lw=2,
         linestyle='--', label='F1-Score')
plt.axvline(x=DEFENSE_NOISE_STD, color='gray', linestyle=':',
            label=f'DEFENSE_NOISE_STD = {DEFENSE_NOISE_STD}')
plt.xlabel('Gaussian Noise Standard Deviation')
plt.ylabel('Score')
plt.title('Figure 4 — Attack Curve: Performance vs. Adversarial Noise Level')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('attack-curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: attack-curve.png')


In [ ]:
# ── Figure 5: Confusion matrix — C2 Under Attack ────────────────────────────
cm_c2 = confusion_matrix(y_test, y_pred_c2)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_c2, annot=True, fmt='d', cmap='OrRd',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title(f'Figure 5 — Confusion Matrix: C2 Under Attack (std={DEFENSE_NOISE_STD})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion-matrix-attack.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion-matrix-attack.png')


---
## Part 4 — Condition 3: Adversarial Training Defense

The training set is **augmented** with perturbed copies of attack-class training samples
(labels preserved as 1). A new Random Forest with identical hyperparameters is trained
on the augmented dataset and evaluated on the **same adversarial test set** used in C2.


In [ ]:
# ── Adversarial training-set generator ──────────────────────────────────────
def create_adversarial_train(X_train, y_train, top5_idx, noise_std, random_state=42):
    """
    Create perturbed copies of attack-class TRAINING samples.
    Labels are kept as 1 — the model learns that disguised attacks are still attacks.
    """
    np.random.seed(random_state)
    attack_mask = (y_train == 1)
    X_attack    = X_train[attack_mask].copy()
    y_attack    = y_train[attack_mask].copy()
    noise       = np.random.normal(0, noise_std,
                                   size=(X_attack.shape[0], len(top5_idx)))
    X_attack[:, top5_idx] += noise
    return X_attack, y_attack

print('create_adversarial_train() defined.')


In [ ]:
# ── Build augmented training set ─────────────────────────────────────────────
X_adv_train, y_adv_train = create_adversarial_train(
    X_train, y_train, top5_idx, noise_std=DEFENSE_NOISE_STD
)

X_train_aug = np.vstack([X_train, X_adv_train])
y_train_aug = np.concatenate([y_train, y_adv_train])

print(f'Original training set:      {X_train.shape}')
print(f'Adversarial samples added:  {X_adv_train.shape}')
print(f'Augmented training set:     {X_train_aug.shape}')
print(f'Class balance — normal: {(y_train_aug==0).sum()}  attack: {(y_train_aug==1).sum()}')


In [ ]:
# ── Train defended Random Forest ─────────────────────────────────────────────
# Same hyperparameters as baseline — ensures a fair comparison.
rf_defended = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_defended.fit(X_train_aug, y_train_aug)
print('Defended model training complete.')


In [ ]:
# ── Evaluate on the SAME adversarial test set used for C2 ────────────────────
y_pred_c3 = rf_defended.predict(X_adv_selected)
y_prob_c3 = rf_defended.predict_proba(X_adv_selected)[:, 1]

acc_c3  = accuracy_score(y_test, y_pred_c3)
prec_c3 = precision_score(y_test, y_pred_c3)
rec_c3  = recall_score(y_test, y_pred_c3)
f1_c3   = f1_score(y_test, y_pred_c3)
auc_c3  = roc_auc_score(y_test, y_prob_c3)

print('=== CONDITION 3: POST-DEFENSE RESULTS ===')
print(f'Accuracy : {acc_c3:.4f}')
print(f'Precision: {prec_c3:.4f}')
print(f'Recall   : {rec_c3:.4f}  (vs C2: {rec_c3 - rec_c2:+.4f}  |  vs C1: {rec_c3 - rec_c1:+.4f})')
print(f'F1-Score : {f1_c3:.4f}')
print(f'AUC-ROC  : {auc_c3:.4f}')


---
## Part 5 — Three-Condition Comparison


In [ ]:
# ── Three-condition comparison table ─────────────────────────────────────────
print(f'Noise level used (DEFENSE_NOISE_STD) = {DEFENSE_NOISE_STD}')
print()
print(f'{"Condition":>25} | {"Accuracy":>10} | {"Precision":>10} | {"Recall":>8} | {"F1":>8} | {"AUC":>8}')
print('-' * 82)
print(f'{"C1: Baseline":>25} | {acc_c1:>10.4f} | {prec_c1:>10.4f} | {rec_c1:>8.4f} | {f1_c1:>8.4f} | {auc_c1:>8.4f}')
print(f'{"C2: Under Attack":>25} | {acc_c2:>10.4f} | {"N/A":>10} | {rec_c2:>8.4f} | {f1_c2:>8.4f} | {auc_c2:>8.4f}')
print(f'{"C3: Post-Defense":>25} | {acc_c3:>10.4f} | {prec_c3:>10.4f} | {rec_c3:>8.4f} | {f1_c3:>8.4f} | {auc_c3:>8.4f}')
print()
print(f'Recall drop     C1 -> C2:  {rec_c1 - rec_c2:+.4f}')
print(f'Recall recovery C2 -> C3:  {rec_c3 - rec_c2:+.4f}')
print(f'Net change      C1 -> C3:  {rec_c3 - rec_c1:+.4f}')


In [ ]:
# ── Figure 6: Grouped bar chart — three-condition comparison ─────────────────
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
c1_vals = [acc_c1, prec_c1, rec_c1, f1_c1, auc_c1]
c2_vals = [acc_c2, 0,       rec_c2, f1_c2, auc_c2]   # precision not tracked for C2
c3_vals = [acc_c3, prec_c3, rec_c3, f1_c3, auc_c3]

x = np.arange(len(metric_labels))
w = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w, c1_vals, w, label='C1: Baseline',     color='steelblue')
b2 = ax.bar(x,     c2_vals, w, label='C2: Under Attack',  color='crimson')
b3 = ax.bar(x + w, c3_vals, w, label='C3: Post-Defense',  color='forestgreen')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Figure 6 — Three-Condition Comparison (noise std = {DEFENSE_NOISE_STD})',
             fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.18)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.annotate('* C2 Precision not tracked (set to 0 in chart)',
            xy=(0.01, 0.01), xycoords='axes fraction', fontsize=8, color='gray')

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.01:
            ax.annotate(f'{h:.3f}',
                        xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('comparison-barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison-barchart.png')


In [ ]:
# ── Figure 7: Confusion matrix — C3 Post-Defense ────────────────────────────
cm_c3 = confusion_matrix(y_test, y_pred_c3)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_c3, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title(f'Figure 7 — Confusion Matrix: C3 Post-Defense (std={DEFENSE_NOISE_STD})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion-matrix-defense.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion-matrix-defense.png')


In [ ]:
# ── Final results summary — copy these into your report ──────────────────────
sep = '=' * 74
print(sep)
print('  FINAL RESULTS  —  copy these values into the report markdown files')
print(sep)
print(f'  DEFENSE_NOISE_STD = {DEFENSE_NOISE_STD}')
print()
print(f'  C1 Baseline  : Acc={acc_c1:.4f}  Prec={prec_c1:.4f}  Rec={rec_c1:.4f}  F1={f1_c1:.4f}  AUC={auc_c1:.4f}')
print(f'  C2 Under Atk : Acc={acc_c2:.4f}  Prec=N/A      Rec={rec_c2:.4f}  F1={f1_c2:.4f}  AUC={auc_c2:.4f}')
print(f'  C3 Post-Def  : Acc={acc_c3:.4f}  Prec={prec_c3:.4f}  Rec={rec_c3:.4f}  F1={f1_c3:.4f}  AUC={auc_c3:.4f}')
print()
print(f'  Recall drop     C1 -> C2 : {rec_c1 - rec_c2:+.4f}')
print(f'  Recall recovery C2 -> C3 : {rec_c3 - rec_c2:+.4f}')
print(f'  Net change      C1 -> C3 : {rec_c3 - rec_c1:+.4f}')
print()
print(f'  Top-5 feature indices : {top5_idx.tolist()}')
print(f'  Top-5 feature names   : {[feature_cols[i] for i in top5_idx]}')
print()
print('  Figures saved in this session:')
for fname in [
    'confusion-matrix-baseline.png',
    'roc-baseline.png',
    'feature-importance-baseline.png',
    'attack-curve.png',
    'confusion-matrix-attack.png',
    'comparison-barchart.png',
    'confusion-matrix-defense.png'
]:
    print(f'    {fname}')
print(sep)
print('  Download figures: Files panel (left sidebar) -> right-click -> Download')
print(sep)
